# Tutorial: RL-02 CartPole Policy Gradients\n\nAudience:\n- Learners who already understand tabular RL and want a first deep RL example.\n\nPrerequisites:\n- Policy gradients, softmax policies, neural networks, and PyTorch basics.\n\nLearning goals:\n- Train a small neural-network policy with REINFORCE on CartPole-v1.\n- Compute discounted returns from sampled trajectories.\n- Keep the experiment CPU-feasible while making the dependency requirements explicit.\n

## Outline\n\n1. Verify the environment and imports\n2. Define a policy network\n3. Collect one episode and compute returns\n4. Train with REINFORCE and inspect learning curves\n

In [ ]:
from __future__ import annotations\n\nimport random\n\nimport numpy as np\n\ntry:\n    import gymnasium as gym\nexcept ImportError as exc:\n    raise ImportError('This notebook requires gymnasium. Install it with `pip install gymnasium[classic-control]`.') from exc\n\ntry:\n    import torch\n    import torch.nn as nn\n    from torch.distributions import Categorical\nexcept ImportError as exc:\n    raise ImportError('This notebook requires PyTorch. Install it with `pip install torch`.') from exc\n\nSEED = 11\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nDEVICE = torch.device('cpu')\nDEVICE\n

## Step 1 - Create the environment\n\nCartPole-v1 is a small classic-control benchmark with a discrete action space and low-dimensional state observations, which makes it a good first policy-gradient environment.\n

In [ ]:
env = gym.make('CartPole-v1')\nstate, info = env.reset(seed=SEED)\nenv.action_space.seed(SEED)\nstate.shape, env.action_space.n\n

## Step 2 - Define a small policy network\n\nThe actor maps the 4-dimensional state to logits over the two discrete actions. A softmax distribution then defines the stochastic policy.\n

In [ ]:
class PolicyNet(nn.Module):\n    def __init__(self, state_dim: int, hidden_dim: int, action_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(state_dim, hidden_dim),\n            nn.Tanh(),\n            nn.Linear(hidden_dim, action_dim),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\npolicy = PolicyNet(state_dim=4, hidden_dim=32, action_dim=2).to(DEVICE)\noptimizer = torch.optim.Adam(policy.parameters(), lr=1e-2)\npolicy\n

## Step 3 - Roll out one episode\n\nREINFORCE uses complete episodes. We store the log-probabilities of sampled actions and the rewards generated along the trajectory.\n

In [ ]:
def sample_episode(env, policy):\n    state, _ = env.reset()\n    log_probs = []\n    rewards = []\n    done = False\n    truncated = False\n    while not (done or truncated):\n        state_tensor = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)\n        logits = policy(state_tensor)\n        dist = Categorical(logits=logits)\n        action = dist.sample()\n        next_state, reward, done, truncated, _ = env.step(action.item())\n        log_probs.append(dist.log_prob(action))\n        rewards.append(reward)\n        state = next_state\n    return log_probs, rewards\n\nlog_probs, rewards = sample_episode(env, policy)\nlen(rewards), sum(rewards)\n

## Step 4 - Compute discounted returns\n\nFor REINFORCE, each policy score term is weighted by the return from that time onward. We normalize returns here to reduce gradient variance.\n

In [ ]:
def discounted_returns(rewards, gamma: float):\n    running = 0.0\n    returns = []\n    for reward in reversed(rewards):\n        running = reward + gamma * running\n        returns.append(running)\n    returns.reverse()\n    returns = torch.tensor(returns, dtype=torch.float32, device=DEVICE)\n    return (returns - returns.mean()) / (returns.std() + 1e-8)\n\ndiscounted_returns(rewards, gamma=0.99)[:5]\n

## Step 5 - Train with REINFORCE\n\nThis is the simplest policy-gradient loop: sample one episode, compute normalized returns, and ascend the policy objective. Keeping the hidden layer small makes the notebook CPU-friendly.\n

In [ ]:
gamma = 0.99\nepisodes = 250\nepisode_lengths = []\nloss_history = []\n\nfor episode in range(episodes):\n    log_probs, rewards = sample_episode(env, policy)\n    returns = discounted_returns(rewards, gamma)\n    policy_loss = torch.stack([-lp * ret for lp, ret in zip(log_probs, returns)]).sum()\n\n    optimizer.zero_grad()\n    policy_loss.backward()\n    optimizer.step()\n\n    episode_lengths.append(len(rewards))\n    loss_history.append(policy_loss.item())\n\nmoving_average = np.convolve(episode_lengths, np.ones(20) / 20, mode='valid')\nepisode_lengths[-10:], moving_average[-5:]\n

## Interpretation\n\nIf training is working, the moving average episode length should trend upward because the pole stays balanced longer. REINFORCE is often noisy, so the per-episode loss is less informative than the return trend.\n

In [ ]:
summary = {\n    'last_10_episode_mean_length': float(np.mean(episode_lengths[-10:])),\n    'best_episode_length': int(np.max(episode_lengths)),\n    'final_moving_average': float(moving_average[-1]) if len(moving_average) else float(np.mean(episode_lengths)),\n}\nsummary\n

## Summary\n\nThis notebook used a neural-network policy to solve a small control environment with REINFORCE. The environment is simple enough for CPU execution, but it already shows the core deep RL loop: sample trajectories, estimate returns, and update the policy by gradient ascent.\n

## Exercises\n\n- Add a learned value baseline and turn the notebook into a simple actor-critic implementation.\n- Compare return normalization against a raw-return version of REINFORCE.\n- Replace REINFORCE with a PPO implementation and compare sample efficiency on the same environment.\n